### Setup

In [9]:
# !pip install torch_geometric numpy torchmetrics # Colab

In [10]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname
pd.set_option("display.max_columns", None)

DATASET         = "bpi_2012"

ROOT_PATH       = dirname(os.getcwd())      
# ROOT_PATH       = "drive/MyDrive/Thesis"    
PROCESSED_PATH  = f"{ROOT_PATH}/data/datasets/processed/{DATASET}"
MODEL_PATH      = f"{ROOT_PATH}/data/datasets/models/{DATASET}"
GRAPHS_PATH     = f"{ROOT_PATH}/data/datasets/graphs/{DATASET}"

In [11]:
with open(f"{ROOT_PATH}/data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

list(datasets_info.keys())

['bpi_2012', 'bpi_2013', 'sp2020', 'BPI20_RequestForPayment']

In [12]:
tab_all = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_all.csv")
tab_all.head()

,CaseID,Activity,org:resource,time:timestamp,(case) AMOUNT_REQ,concept:name,lifecycle:transition
0,1,A_SUBMITTED-COMPLETE,Value 1,0.000000,9.903538,A_SUBMITTED,COMPLETE
1,1,A_PARTLYSUBMITTED-COMPLETE,Value 1,0.288182,9.903538,A_PARTLYSUBMITTED,COMPLETE
2,1,A_PREACCEPTED-COMPLETE,Value 1,3.995629,9.903538,A_PREACCEPTED,COMPLETE
3,1,W_Completeren aanvraag-SCHEDULE,Value 1,4.013297,9.903538,W_Completeren aanvraag,SCHEDULE
4,1,W_Completeren aanvraag-START,Value 2,10.583623,9.903538,W_Completeren aanvraag,START


In [13]:
dataset_info = datasets_info[DATASET]
dataset_info

{'categorical': ['Activity',
  'org:resource',
  'concept:name',
  'lifecycle:transition'],
 'numerical': ['time:timestamp', '(case) AMOUNT_REQ']}

In [14]:
tab_train = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_train.csv")
tab_valid = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_valid.csv")
tab_test = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_test.csv")

In [15]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

print(categorical_columns)
print(real_value_columns)

['Activity', 'org:resource', 'concept:name', 'lifecycle:transition']
['time:timestamp', '(case) AMOUNT_REQ']


In [16]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")

In [17]:
if DATASET == "sp2020":
    tab_all["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_all["REPAIR_IN_TIME_5D"].values]
    tab_train["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_train["REPAIR_IN_TIME_5D"].values]
    tab_valid["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_valid["REPAIR_IN_TIME_5D"].values]
    tab_test["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_test["REPAIR_IN_TIME_5D"].values]

In [18]:
from math import log

def log_norm(x):
    return log(x+1)

if DATASET == "BPI20_RequestForPayment_CZ":
    tab_all["case:RequestedAmount"] = tab_all["case:RequestedAmount"].apply(log_norm)
    tab_train["case:RequestedAmount"] = tab_train["case:RequestedAmount"].apply(log_norm)
    tab_valid["case:RequestedAmount"] = tab_valid["case:RequestedAmount"].apply(log_norm)
    tab_test["case:RequestedAmount"] = tab_test["case:RequestedAmount"].apply(log_norm)
   

### Prepare the graphs

In [19]:
from utils import get_case_ids, get_one_hot_encodings
from torch import tensor,int64, float32
from torch_geometric.data import HeteroData


In [20]:
import sklearn.preprocessing

""" Create onehot encoder for the column types"""
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = dataset[key].unique()                   # Get all the values a key could have
    if key == "Activity":                           # Add the END node
        datas = np.union1d(datas, ["END"])         
    datas = datas.astype(np.str_)                   # Make everything a string
    datas = datas.reshape([len(datas), 1])          # Add a dimension
    onehot = sklearn.preprocessing.OneHotEncoder()  # Learn and return
    onehot.fit(datas)
    return onehot

In [21]:
ONE_HOT_ENCODERS = {k: get_one_hot_encoder(tab_all, k) for k in categorical_columns}
ONE_HOT_ENCODERS

{'Activity': OneHotEncoder(),
 'org:resource': OneHotEncoder(),
 'concept:name': OneHotEncoder(),
 'lifecycle:transition': OneHotEncoder()}

In [22]:
""" Normalize the timestamps (relative duration from the start) """
def add_new_timestamp(trace: pd.DataFrame):
    times = list(trace["time:timestamp"].copy())
    for i in range(1,len(times)):
        times[i] = times[i] - times[0]
    times[0] = 0.
    trace2 = trace.copy()
    trace2["time:timestamp"] = times 
    return trace2

In [23]:
""" Get trace columns and encode it into the correct tensors """
def get_node_features(trace: pd.DataFrame, cat_features, real_features) -> dict:
    columns_static = [c for c in trace if len(set(trace[c])) == 1]

    res = {}

    for key in trace:
        values = trace[key].values
        # Categorical feature handling
        if key in cat_features:
            onehot_encoder = ONE_HOT_ENCODERS[key]
            values = values.astype(np.str_)
            if key not in columns_static:
                try:
                    res[key] = tensor(
                        get_one_hot_encodings(onehot_encoder, values),
                        dtype=float32
                    )
                except ValueError:
                    print("Error in the encoding")
                    print(key)
                    print(values)
            else:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, np.array([values[0]])),
                    dtype=float32
                )
        # Numerical features handling
        elif key in real_features:
            if key not in columns_static:
                res[key] = tensor(values,  dtype=float32)
            else:
                res[key] = tensor([values[0]], dtype=float32)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res

In [25]:
""" Get the edge type that links different types of nodes """
def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
    # activities indexes
    for k in keys:
        if len(node_features[k]) != 1:      # Only dynamic features
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [26]:
from tqdm.notebook import tqdm
from copy import copy
import torch

from torch_geometric.transforms import ToUndirected
UNDIRECT_TRANSFORMATION = ToUndirected()

""" Builds n-2 HG out of a trace """
def build_prefixes_graph_from_trace(trace, cat_features, real_features):
    X = []  # graphs
    trace = add_new_timestamp(trace)  # Normalize time
    node_features = get_node_features(trace, cat_features, real_features)
    
    for prefix in range(1, len(trace)-1):
        # For each trace get the prefix slice
        G = HeteroData()
        for k in node_features:
            G[k].x = node_features[k][:(prefix+1)] 

        # Build the edges
        edges_indexes = compute_edges_indexs(node_features, prefix_len=prefix+1)

        # Convert in PyG
        for k in edges_indexes:
            ce = [[], []]
            for i in range(len(edges_indexes[k])):
                ce[0].append(edges_indexes[k][i][0])
                ce[1].append(edges_indexes[k][i][1])
            edges_indexes[k] = ce

        # Assign edges to graph
        for k in edges_indexes:
            G[k].edge_index = tensor(edges_indexes[k], dtype=torch.long)

        # Build labels (aka the targets)
        G.y = {}
        for k in node_features:
            if len(node_features[k]) != 1:  # dynamic
                if k in cat_features:
                    G.y[k] = torch.max(node_features[k][prefix+1], 0)[1].reshape(1,-1)[0].detach().clone()
                else:
                    G.y[k] = node_features[k][prefix+1].reshape(1,-1)[0].detach().clone()
            else:                           # static 
                if k in cat_features:
                    G.y[k] = torch.max(node_features[k][0], 0)[1].reshape(1,-1)[0].detach().clone()
                else:
                    G.y[k] = node_features[k][0].reshape(1,-1)[0].detach().clone()
        
        G = UNDIRECT_TRANSFORMATION(G)
        G.is_end = torch.tensor([False])
        X.append(G)

    # Final graph (with END as label)
    G_end = HeteroData()
    for k in node_features:
        G_end[k].x = node_features[k]  # full trace, no END node appended

    edges_indexes = compute_edges_indexs(node_features, prefix_len=len(trace))
    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    for k in edges_indexes:
        G_end[k].edge_index = torch.tensor(edges_indexes[k], dtype=torch.long)

    # Assegnazione dinamica dei target END
    G_end.y = {}
    for k in node_features:
        if k in cat_features:
            # Trova l'indice della classe "END" dinamicamente per questa specifica categoria
            if k == "Activity":
                end_idx = int(ONE_HOT_ENCODERS[k].transform([["END"]]).argmax())
                G_end.y[k] = torch.tensor([end_idx], dtype=torch.long)
            else:
                G_end.y[k] = torch.max(node_features[k][-1], 0)[1].reshape(1)
        else:
            # Per le feature reali (timestamp), copiamo il valore dell'ultimo evento registrato.
            last_real_value = node_features[k][-1].reshape(1,-1)[0].detach().clone()
            G_end.y[k] = last_real_value

    G_end = UNDIRECT_TRANSFORMATION(G_end)
    G_end.is_end = torch.tensor([True])
    X.append(G_end)

    return X

In [27]:
""" Adds end activity to the ground truth"""
def add_end_activity_ground(t):
    end_feat = torch.tensor(
        ONE_HOT_ENCODERS["Activity"].transform([["END"]]).toarray(),
        dtype=torch.float32
    )
    t["Activity"] = torch.cat([t["Activity"], end_feat], dim=0)
    return t

In [28]:
""" Build (2-prefix, ground truth) tuple"""
def build_test_graph_from_trace(trace, cat_features, real_features):
    trace = add_new_timestamp(trace) # Normalize time
    node_features = get_node_features(trace, cat_features, real_features) # Get node features

    # Extracting a length two prefix from every trace
    G = HeteroData()
    for k in node_features:
        G[k].x = node_features[k][:2]

    # Build the edges
    edges_indexes = compute_edges_indexs(node_features, prefix_len=2)

    # Convert in PyG
    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    # Assign edges to the graph
    for k in edges_indexes:
        G[k].edge_index = tensor(edges_indexes[k], dtype=torch.long)

    G = UNDIRECT_TRANSFORMATION(G)

    # Ground truth: full activity sequence as class indexes
    ground_truth = {}

    for k in node_features:
        ground_truth[k] = node_features[k].detach().clone()

    ground_truth = add_end_activity_ground(ground_truth)

    return (G, ground_truth)

## Create the graph datasets

In [29]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [30]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

7852
2617
2618


In [31]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [32]:
print("Preparing training dataset...")

X_train = []

for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )

        N_GRAPHS = len(graphs)

        for j in range(N_GRAPHS):
            X_train.append(graphs[j])

Preparing training dataset...


  0%|          | 0/7852 [00:00<?, ?it/s]

In [ ]:
torch.save(X_train, f"{GRAPHS_PATH}/train_set.pt")
print("Train Graphs created!\n\n")

In [ ]:
import random
g_train = random.choice([g for g in X_train if g.is_end.item()])

In [ ]:
del X_train

In [ ]:
print("Preparing validation dataset...")

X_val = []

for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )

        N_GRAPHS = len(graphs)


        for j in range(N_GRAPHS):
            X_val.append(graphs[j])

In [ ]:
torch.save(X_val, f"{GRAPHS_PATH}/validation_set.pt")
print("Val Graph created!\n\n")

In [ ]:
g_val = random.choice([g for g in X_val if g.is_end.item()])

In [ ]:
del X_val

In [ ]:
print("Preparing test dataset...")

X_test = []

for i in tqdm(range(len(case_test_ids))):
    trace = (
        tab_test.query(f"CaseID == '{case_test_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        pair = build_test_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )

        X_test.append(pair)

In [ ]:
torch.save(X_test, f"{GRAPHS_PATH}/test_set.pt")
print("Test Graphs created!\n\n")

In [ ]:
g_test, t_test = random.choice(X_test)

In [ ]:
del X_test

In [ ]:
# Overview of the graph structure
print(g_train)
print(g_val)
print(g_test)
print(t_test)  # this is the ground truth dict, not a graph